In [6]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("MiApp") \
    .master("local[*]") \
    .getOrCreate()

sc = spark.sparkContext

working_dir = "/opt/app/working_dir/"
rdd = sc.textFile(working_dir + "constitution.txt")

print("Primeras 3 líneas:", rdd.take(3))
print("Cantidad de líneas:", rdd.count())

splitted_lines = rdd.map(lambda line: line.split(" "))
print("Ejemplo con map:", splitted_lines.take(3))

words_rdd = rdd.flatMap(lambda line: line.strip().split(" "))
print("Ejemplo con flatMap:", words_rdd.take(10))
print("Total de palabras (ingenuo):", words_rdd.count())

words_rdd = rdd.flatMap(lambda line: line.strip().split(" ")) \
               .filter(lambda w: w != "")
print("Total de palabras (ajustado):", words_rdd.count())

longest = words_rdd.reduce(lambda a, b: a if len(a) > len(b) else b)
print("Palabra más larga:", longest, "(", len(longest), "caracteres )")

words_rdd = words_rdd.filter(lambda w: w.isalnum())
print("Muestra alfanumérica:", words_rdd.take(10))
print("Total de palabras alfanuméricas:", words_rdd.count())

keyval_rdd = words_rdd.map(lambda word: (word.lower(), 1))
wordcount = keyval_rdd.reduceByKey(lambda a, b: a + b)

sorted_wc = wordcount.map(lambda x: (x[1], x[0])) \
                     .sortByKey(ascending=False)
print("Top 5 palabras más repetidas:", sorted_wc.take(5))

stopwords = set([
    "the", "of", "and", "to", "a", "in", "is", "it", "that", "as", 
    "for", "with", "was", "on", "be", "by", "an", "this", "which", "or"
])

filtered_wc = wordcount.filter(lambda x: x[0] not in stopwords)
sorted_filtered = filtered_wc.map(lambda x: (x[1], x[0])) \
                             .sortByKey(ascending=False)
print("Top 5 palabras más repetidas (sin stopwords):", sorted_filtered.take(5))


Primeras 3 líneas: ['We the People of the United States, in Order to form a more perfect ', 'Union, establish Justice, insure domestic Tranquility, provide for the ', 'common defence, promote the general Welfare, and secure the Blessings of ']
Cantidad de líneas: 649
Ejemplo con map: [['We', 'the', 'People', 'of', 'the', 'United', 'States,', 'in', 'Order', 'to', 'form', 'a', 'more', 'perfect', ''], ['Union,', 'establish', 'Justice,', 'insure', 'domestic', 'Tranquility,', 'provide', 'for', 'the', ''], ['common', 'defence,', 'promote', 'the', 'general', 'Welfare,', 'and', 'secure', 'the', 'Blessings', 'of', '']]
Ejemplo con flatMap: ['We', 'the', 'People', 'of', 'the', 'United', 'States,', 'in', 'Order', 'to']
Total de palabras (ingenuo): 7763
Total de palabras (ajustado): 7623
Palabra más larga: Representatives, ( 16 caracteres )
Muestra alfanumérica: ['We', 'the', 'People', 'of', 'the', 'United', 'in', 'Order', 'to', 'form']
Total de palabras alfanuméricas: 6701
Top 5 palabras más repe

## Discusión de resultados

### 1. Tipo de datos en el RDD inicial
Al cargar el archivo con sc.textFile cada elemento del RDD corresponde a una línea de texto completa del archivo. Al hacer take(3) aparecen strings completos. El RDD almacena elementos de tipo str.

### 2. Cantidad de líneas en el documento
El archivo tiene 649 líneas de texto. Spark cuenta cada salto de línea como un elemento distinto.

### 3. ¿cumple con nuestro propósito? ¿conoceremos cuántas palabras hay en el texto?
Con map obtuve listas de palabras, pero no un RDD de palabras individuales. Con flatMap logré transformar cada línea en múltiples palabras. El conteo ingenuo de 7763 resultó inflado, porque split(" ") genera cadenas vacías cuando hay espacios al inicio o al final de una línea. Tras ajustar y filtrar esas cadenas vacías, el total real fue de 7623 palabras. Esto muestra la importancia de limpiar los datos.

### 4. Palabra más larga
El reduce devolvió Representatives, con 16 caracteres. Spark considera la coma como parte de la palabra, porque el separador fueron solo los espacios. Al filtrar únicamente palabras alfanuméricas el corpus quedó más limpio, y el total de palabras bajó a 6701.

### 5. Palabras más frecuentes
El top 5 fue:
the → 726  
of → 493  
shall → 293  
and → 262  
to → 201  

Esto muestra un patrón común en textos en inglés, donde palabras funcionales dominan la frecuencia. Estas no aportan significado semántico fuerte y suelen considerarse stopwords.

### 6. Palabras más frecuentes excluyendo stopwords
Después de eliminar stopwords, el top 5 cambió a:
shall → 293  
united → 85  
any → 79  
president → 72  
have → 63  

Aquí aparecen términos más significativos para el contenido del documento. Palabras como shall, united y president muestran conceptos relevantes dentro de la constitución.


